In [1]:
# ================================
# Import Libraries
# ================================

import numpy as np
import string
import pickle

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding, LSTM, Dense

In [2]:
# ================================
# Load Dataset
# ================================

with open("train.txt", "r", encoding="utf-8") as f:
    train_text = f.read()

with open("test.txt", "r", encoding="utf-8") as f:
    test_text = f.read()

print("Train Characters:", len(train_text))
print("Test Characters:", len(test_text))

print("\nTrain Sample:")
print(train_text[:500])

Train Characters: 10810591
Test Characters: 1122901

Train Sample:
" 
 = 2013 – 14 York City F.C. season = 
 
 The 2013 – 14 season was the <unk> season of competitive association football and 77th season in the Football League played by York City Football Club , a professional football club based in York , North Yorkshire , England . Their 17th @-@ place finish in 2012 – 13 meant it was their second consecutive season in League Two . The season ran from 1 July 2013 to 30 June 2014 . 
 Nigel Worthington , starting his first full season as York manager , made ei


In [3]:
# ================================
# Text Cleaning
# ================================

train_text = train_text.lower()
test_text = test_text.lower()

translator = str.maketrans(
    "",
    "",
    string.punctuation
)

train_text = train_text.translate(translator)
test_text = test_text.translate(translator)

print("\nCleaned Train Sample:")
print(train_text[:500])


Cleaned Train Sample:
 
  2013 – 14 york city fc season  
 
 the 2013 – 14 season was the unk season of competitive association football and 77th season in the football league played by york city football club  a professional football club based in york  north yorkshire  england  their 17th  place finish in 2012 – 13 meant it was their second consecutive season in league two  the season ran from 1 july 2013 to 30 june 2014  
 nigel worthington  starting his first full season as york manager  made eight permanent summ


In [4]:
# ================================
# Text Cleaning
# ================================

train_text = train_text.lower()
test_text = test_text.lower()

translator = str.maketrans(
    "",
    "",
    string.punctuation
)

train_text = train_text.translate(translator)
test_text = test_text.translate(translator)

print("\nCleaned Train Sample:")
print(train_text[:500])


Cleaned Train Sample:
 
  2013 – 14 york city fc season  
 
 the 2013 – 14 season was the unk season of competitive association football and 77th season in the football league played by york city football club  a professional football club based in york  north yorkshire  england  their 17th  place finish in 2012 – 13 meant it was their second consecutive season in league two  the season ran from 1 july 2013 to 30 june 2014  
 nigel worthington  starting his first full season as york manager  made eight permanent summ


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

Tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

Tokenizer.fit_on_texts([train_text])

In [6]:
# ================================
# Convert Text to Sequences
# ================================

train_sequences = Tokenizer.texts_to_sequences(
    [train_text]
)[0]

test_sequences = Tokenizer.texts_to_sequences(
    [test_text]
)[0]

print("\nNumber of Train Tokens:")
print(len(train_sequences))

print("\nNumber of Test Tokens:")
print(len(test_sequences))


Number of Train Tokens:
1756108

Number of Test Tokens:
184238


In [7]:
# ================================
# Display Examples
# ================================

words = train_text.split()

for i in range(10):

    print(
        "\nWord:",
        words[i],
        " | Token:",
        train_sequences[i]
    )


Word: 2013  | Token: 400

Word: –  | Token: 34

Word: 14  | Token: 329

Word: york  | Token: 244

Word: city  | Token: 81

Word: fc  | Token: 6079

Word: season  | Token: 78

Word: the  | Token: 2

Word: 2013  | Token: 400

Word: –  | Token: 34


In [8]:
# ================================
# Create Training Data
# ================================

X = []
y = []

max_len = 20

for i in range(1, len(train_sequences)):

    start = max(0, i - max_len)

    X.append(
        train_sequences[start:i]
    )

    y.append(
        train_sequences[i]
    )

print("\nTotal Training Samples:")
print(len(X))


Total Training Samples:
1756107


In [9]:
# ================================
# Padding
# ================================

X = pad_sequences(
    X,
    maxlen=max_len,
    padding="pre"
)

y = np.array(y)

print("\nX Shape:")
print(X.shape)

print("\nY Shape:")
print(y.shape)


X Shape:
(1756107, 20)

Y Shape:
(1756107,)


In [10]:
# ================================
# Create Test Data
# ================================

X_test = []
y_test = []

for i in range(1, len(test_sequences)):

    start = max(0, i - max_len)

    X_test.append(
        test_sequences[start:i]
    )

    y_test.append(
        test_sequences[i]
    )

print("\nTest Samples:")
print(len(X_test))


Test Samples:
184237


In [11]:
# ================================
# Padding Test Data
# ================================

X_test = pad_sequences(
    X_test,
    maxlen=max_len,
    padding="pre"
)

y_test = np.array(y_test)

print("\nX Test Shape:")
print(X_test.shape)

print("\nY Test Shape:")
print(y_test.shape)


X Test Shape:
(184237, 20)

Y Test Shape:
(184237,)


In [12]:
# ================================
# Build LSTM Model
# ================================
vocab_size = len(Tokenizer.word_index) + 1

embedding_dim = 100
lstm_units = 128

lstm_model = Sequential([
    
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim
    ),

    LSTM(
        lstm_units
    ),

    Dense(
        vocab_size,
        activation="softmax"
    )
])

print(lstm_model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [13]:
# ================================
# Compile Model
# ================================

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
# ================================
# Train Model
# ================================

history = lstm_model.fit(
    X,
    y,
    epochs=2,
    batch_size=64,
    validation_split=0.1
)    

Epoch 1/2
24696/24696 ━━━━━━━━━━━━━━━━━━━━ 15752s 638ms/step - accuracy: 0.1506 - loss: 5.8794 - val_accuracy: 0.1691 - val_loss: 5.7863
Epoch 2/2
24696/24696 ━━━━━━━━━━━━━━━━━━━━ 35002s 1s/step - accuracy: 0.1857 - loss: 5.2347 - val_accuracy: 0.1777 - val_loss: 5.6373


In [15]:
# ================================
# Evaluate Model
# ================================

test_loss, test_accuracy = lstm_model.evaluate(
    X_test,
    
    y_test,
    batch_size=64
)

print("\nTest Loss:")
print(test_loss)

print("\nTest Accuracy:")
print(test_accuracy)

2879/2879 ━━━━━━━━━━━━━━━━━━━━ 262s 91ms/step - accuracy: 0.1748 - loss: 5.2548

Test Loss:
5.254756927490234

Test Accuracy:
0.17478030920028687


In [16]:
# ================================
# Evaluate Model
# ================================

test_loss, test_accuracy = lstm_model.evaluate(
    X_test,
    y_test,
    batch_size=64
)

print("\nTest Loss:")
print(test_loss)

print("\nTest Accuracy:")
print(test_accuracy)

2879/2879 ━━━━━━━━━━━━━━━━━━━━ 264s 92ms/step - accuracy: 0.1748 - loss: 5.2548

Test Loss:
5.254756927490234

Test Accuracy:
0.17478030920028687


In [17]:
# ================================
# Save Model
# ================================

lstm_model.save(
    "lstm_wikitext2_model.h5"
)

print("\nModel Saved Successfully!")


Model Saved Successfully!


In [18]:
# ================================
# Save Tokenizer
# ================================

with open(
    "tokenizer.pkl",
    "wb"
) as f:

    pickle.dump(
        Tokenizer,
        f
    )

print("Tokenizer Saved Successfully!")

Tokenizer Saved Successfully!


In [19]:
# ================================
# Save Max Length
# ================================

with open(
    "max_len.pkl",
    "wb"
) as f:

    pickle.dump(
        max_len,
        f
    )

print("Max Length Saved Successfully!")

Max Length Saved Successfully!


In [20]:
# ================================
# Create Index to Word Dictionary
# ================================

index_to_word = {}

for word, index in Tokenizer.word_index.items():

    if index < vocab_size:

        index_to_word[index] = word

In [21]:
# ================================
# Top-5 Next Word Predictor
# ================================

def predictor(
    model,
    tokenizer,
    text,
    max_len,
    top_n=5
):

    text = text.lower()

    text = text.translate(
        translator
    )

    seq = tokenizer.texts_to_sequences(
        [text]
    )[0]

    if len(seq) == 0:

        return []

    # Keep last 20 words
    seq = seq[-max_len:]

    seq = pad_sequences(
        [seq],
        maxlen=max_len,
        padding="pre"
    )

    pred = model.predict(
        seq,
        verbose=0
    )[0]

    # Top-N indexes
    top_indices = np.argsort(
        pred
    )[-top_n:][::-1]

    predictions = []

    for index in top_indices:

        word = index_to_word.get(
            index,
            ""
        )

        if word:

            predictions.append(
                word
            )

    return predictions

In [22]:
# ================================
# Test Prediction
# ================================

seed = "what is"

predictions = predictor(
    lstm_model,
    Tokenizer,
    seed,
    max_len,
    top_n=5
)

print("\nInput:")
print(seed)

print("\nNext Word Suggestions:")

for i, word in enumerate(
    predictions,
    1
):

    print(
        i,
        ".",
        word
    )


Input:
what is

Next Word Suggestions:
1 . a
2 . <OOV>
3 . the
4 . not
5 . played


In [23]:
# ================================
# Generate Text
# ================================

def generate_text(
    model,
    tokenizer,
    seed_text,
    max_len,
    n_words
):

    generated_text = seed_text

    for _ in range(n_words):

        predictions = predictor(
            model,
            tokenizer,
            generated_text,
            max_len,
            top_n=1
        )

        if not predictions:

            break

        next_word = predictions[0]

        generated_text += (
            " " + next_word
        )

    return generated_text

In [24]:
# ================================
# Generate Sentence
# ================================

seed = "what is"

generated_text = generate_text(
    lstm_model,
    Tokenizer,
    seed,
    max_len,
    10
)

print("\nGenerated Text:")
print(generated_text)


Generated Text:
what is a <OOV> <OOV> and <OOV> <OOV> <OOV> <OOV> <OOV> <OOV>
